# Laya vs Laya MoE — head-to-head comparison

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vishalmysore/layaMOE/blob/main/notebooks/laya_vs_moe_comparison.ipynb)

This notebook runs two models on the same **108 hand-labeled cases (9 domains, 312 answers)** and compares them:

| Model | What it is |
|---|---|
| **Original Laya** | [`convaiinnovations/laya-typed-decisions`](https://huggingface.co/convaiinnovations/laya-typed-decisions), loaded with the official `laya` package, unchanged |
| **Laya MoE** | Same frozen encoder + domain **expert heads** ([`VishalMysore/layaMOE`](https://huggingface.co/VishalMysore/layaMOE)) + a router; unrouted text falls back to the original head |
| *MoE (oracle)* | Upper bound: the expert that owns each domain answers, as if routing were perfect |

Experts: **`safety`** (agent-guardrails, content-moderation) and **`customer_ops`** (support-tickets, delivery-exceptions, email-triage). The other 4 domains have no expert, so they test whether the MoE *hurts* anything outside its specialty.

Runs on the free CPU runtime (roughly 10–15 min for the full eval). No API keys needed.

## 1. Setup

In [ ]:
%%capture
!pip install -q laya==0.3.4 safetensors huggingface_hub scipy

In [ ]:
import os, sys, subprocess
REPO = "layaMOE"
if os.path.exists("laya_moe") and os.path.exists("data/eval"):   # running from inside a local checkout
    REPO = "."
elif os.path.exists(os.path.join("..", "laya_moe")):                 # running from notebooks/ in a local checkout
    REPO = ".."
elif not os.path.exists(os.path.join(REPO, "laya_moe")):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/vishalmysore/layaMOE.git", REPO], check=True)
sys.path.insert(0, os.path.abspath(REPO))

import glob, json, time
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
import laya
from huggingface_hub import snapshot_download
from laya_moe.moe import MoEAgent

torch.set_grad_enabled(False)
torch.manual_seed(0)
BASE = "convaiinnovations/laya-typed-decisions"
print("torch", torch.__version__, "| CPU threads", torch.get_num_threads())

## 2. Load the eval set
Each domain file has a shared set of typed questions (`choice`, `score`, `noul` = yes/no probability) and 12 labeled cases. Some labels are marked `hard` (judgement calls) and are also reported separately.

In [ ]:
docs = []
for p in sorted(glob.glob(os.path.join(REPO, "data", "eval", "*.json"))):
    if p.endswith("index.json"):
        continue
    docs.append(json.load(open(p, encoding="utf-8")))

print(f"{'domain':22} cases  questions")
for d in docs:
    print(f"{d['domain']:22} {len(d['cases']):5d}  " + ", ".join(f"{q} ({s['type']})" for q, s in d["questions"].items()))
print("total cases:", sum(len(d["cases"]) for d in docs),
      "| total answers:", sum(len(d["cases"]) * len(d["questions"]) for d in docs))

## 3. Load both models
Both run on CPU in fp32 so the numbers are reproducible and match the project's reported results.
The MoE shares one frozen encoder (~395M params); each expert is only a ~26.5M-param head.

In [ ]:
t = time.time()
original = laya.load(BASE, device="cpu")
original.model.eval()
print(f"original Laya loaded in {time.time() - t:.0f}s")

expert_dir = snapshot_download("VishalMysore/layaMOE")
t = time.time()
moe = MoEAgent(experts=[os.path.join(expert_dir, "safety"), os.path.join(expert_dir, "customer_ops")], threshold=0.3)
print(f"Laya MoE loaded in {time.time() - t:.0f}s")

owner = {dom: name for name, (_, meta) in moe.experts.items() for dom in meta["domains"]}
for name, (head, meta) in moe.experts.items():
    n = sum(p.numel() for p in head.parameters())
    print(f"  expert {name:13} {n/1e6:5.1f}M params  domains={meta['domains']}  "
          f"val acc (synthetic) {meta['train']['val_acc_base_head']:.1%} -> {meta['train']['val_acc']:.1%}")

## 4. Run every case through both models
For each case we record the original answer, the MoE answer (with the router's choice), and the oracle-expert answer.

In [ ]:
def top(probs):
    return max(probs, key=probs.get)

def score_answer(spec, ans, want):
    """-> (prediction, correct, within_one, confidence_of_prediction)"""
    if spec["type"] == "noul":
        p = ans["noul"]; got = p >= 0.5
        return got, got == want, got == want, max(p, 1 - p)
    got = top(ans["probabilities"]); conf = ans["probabilities"][got]
    if spec["type"] == "score":
        got = int(got)
        return got, got == want, abs(got - want) <= 1, conf
    return got, got == want, got == want, conf

rows, routes, timing = [], [], {"original": 0.0, "moe": 0.0}
t0 = time.time()
for d in docs:
    dom, qs = d["domain"], d["questions"]
    for case in d["cases"]:
        s = case["state"]
        t = time.time(); a_orig = original.system_one(s, qs)["answers"]; timing["original"] += time.time() - t
        t = time.time(); chosen, routing = moe.route(s); a_moe = moe.answer_with(chosen, s, qs); timing["moe"] += time.time() - t
        a_orc = moe.answer_with(owner[dom], s, qs) if dom in owner else a_orig
        routes.append({"domain": dom, "id": case["id"], "routed_to": chosen, "router_top": routing["top"],
                       "router_p": routing["probabilities"][routing["top"]]})
        for q, spec in qs.items():
            want = case["expected"][q]
            r = {"domain": dom, "id": case["id"], "question": q, "qtype": spec["type"], "expected": want,
                 "hard": q in case.get("hard", []), "covered": dom in owner, "routed_to": chosen}
            for m, a in (("original", a_orig), ("moe", a_moe), ("oracle", a_orc)):
                r[f"{m}_pred"], r[f"{m}_ok"], r[f"{m}_w1"], r[f"{m}_conf"] = score_answer(spec, a[q], want)
            rows.append(r)
    print(f"  {dom:22} done  ({time.time() - t0:.0f}s)")

df = pd.DataFrame(rows)
for c in [c for c in df.columns if c.endswith(("_ok", "_w1"))] + ["hard", "covered"]:
    df[c] = df[c].astype(bool)
rt = pd.DataFrame(routes)
n_cases = len(rt)
print(f"\n{len(df)} answers from {n_cases} cases")
print(f"avg latency per case: original {1000*timing['original']/n_cases:.0f} ms | "
      f"MoE (route + answer) {1000*timing['moe']/n_cases:.0f} ms")

### Sanity check
When the router sends a case to `general`, the MoE uses the untouched base head, so its answers must match the original Laya's. This confirms the comparison is apples-to-apples.

In [ ]:
gen = df[df.routed_to == "general"]
same = (gen.original_pred == gen.moe_pred).mean() if len(gen) else float("nan")
print(f"{len(gen)} answers routed to 'general': MoE == original on {same:.1%} of them")
assert len(gen) == 0 or same == 1.0, "general head should reproduce the original model exactly"

## 5. Headline results

In [ ]:
def acc_row(g):
    return pd.Series({"answers": len(g), "original": g.original_ok.mean(), "moe": g.moe_ok.mean(),
                      "oracle": g.oracle_ok.mean(), "Δ moe − original": g.moe_ok.mean() - g.original_ok.mean()})

def acc_table(frame, by):
    return pd.DataFrame({k: acc_row(g) for k, g in frame.groupby(by)}).T

summary = pd.DataFrame({
    "ALL": acc_row(df),
    "covered domains (have an expert)": acc_row(df[df.covered]),
    "other domains (no expert)": acc_row(df[~df.covered]),
    "clear labels only": acc_row(df[~df.hard]),
}).T
fmt = {"original": "{:.1%}", "moe": "{:.1%}", "oracle": "{:.1%}", "Δ moe − original": "{:+.1%}", "answers": "{:.0f}"}
display(summary.style.format(fmt).set_caption("Accuracy"))

In [ ]:
by_dom = acc_table(df, "domain")
by_dom["expert"] = [owner.get(d, "—") for d in by_dom.index]
by_dom = by_dom.sort_values("Δ moe − original", ascending=False)
display(by_dom.style.format(fmt).background_gradient(subset=["Δ moe − original"], cmap="RdYlGn", vmin=-0.3, vmax=0.3)
        .set_caption("Accuracy by domain"))

by_type = acc_table(df, "qtype")
display(by_type.style.format(fmt).set_caption("Accuracy by question type"))
w1 = df[df.qtype == "score"][["original_w1", "moe_w1", "oracle_w1"]].mean()
print("score questions, within one level:  " + "  ".join(f"{k[:-3]} {v:.1%}" for k, v in w1.items()))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.8))
x = np.arange(len(by_dom)); w = 0.27
labels = {"original": "Original Laya", "moe": "Laya MoE", "oracle": "MoE (oracle routing)"}
for i, (m, c) in enumerate([("original", "#9aa5b1"), ("moe", "#2f6fdf"), ("oracle", "#8fb8ff")]):
    ax.bar(x + (i - 1) * w, by_dom[m].astype(float), w, label=labels[m], color=c)
ax.set_xticks(x, [f"{d}\n[{e}]" for d, e in zip(by_dom.index, by_dom.expert)], fontsize=8)
ax.set_ylim(0, 1.05); ax.set_ylabel("accuracy"); ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.axhline(df.original_ok.mean(), color="#9aa5b1", ls="--", lw=1)
ax.axhline(df.moe_ok.mean(), color="#2f6fdf", ls="--", lw=1)
ax.set_title("Accuracy per domain (dashed lines = overall)"); ax.legend(frameon=False, ncol=3, loc="upper right")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

## 6. Is the difference real? (paired statistics)
Both models answer the *same* 312 questions, so we use paired tests:
* **McNemar's exact test** on the discordant pairs (questions only one model got right).
* **Paired bootstrap** over *cases* (answers within a case are correlated) for a 95% CI on the accuracy gain.

In [ ]:
from scipy.stats import binomtest

def paired_stats(frame, a="original_ok", b="moe_ok", n_boot=5000, seed=0):
    fixed = int((~frame[a] & frame[b]).sum())   # b right, a wrong
    broke = int((frame[a] & ~frame[b]).sum())   # a right, b wrong
    p = binomtest(fixed, fixed + broke, 0.5).pvalue if fixed + broke else 1.0
    g = frame.groupby("id").agg(sa=(a, "sum"), sb=(b, "sum"), n=(a, "size"))
    idx = np.random.default_rng(seed).integers(0, len(g), size=(n_boot, len(g)))
    deltas = (g.sb.values[idx].sum(1) - g.sa.values[idx].sum(1)) / g.n.values[idx].sum(1)
    lo, hi = np.percentile(deltas, [2.5, 97.5])
    return {"fixed": fixed, "broke": broke, "delta": frame[b].mean() - frame[a].mean(), "ci95": (lo, hi), "p": p}

for name, sel in [("ALL", df), ("covered domains", df[df.covered]), ("other domains", df[~df.covered])]:
    s = paired_stats(sel)
    print(f"{name:16}  Δ = {s['delta']:+.1%}  95% CI [{s['ci95'][0]:+.1%}, {s['ci95'][1]:+.1%}]  "
          f"fixed {s['fixed']:3d} / broke {s['broke']:3d}  McNemar p = {s['p']:.4f}")

## 7. Router behaviour
Rows = true domain, columns = which head the router picked. Ideally `agent-guardrails`/`content-moderation` → `safety`, `support-tickets`/`delivery-exceptions`/`email-triage` → `customer_ops`, and everything else → `general`.

In [ ]:
expected_route = rt.domain.map(lambda d: owner.get(d, "general"))
print(f"routing accuracy: {(rt.routed_to == expected_route).mean():.1%}  "
      f"({(rt.routed_to == expected_route).sum()}/{len(rt)} cases)")
conf_mat = pd.crosstab(rt.domain, rt.routed_to).reindex(columns=["safety", "customer_ops", "general"], fill_value=0)
display(conf_mat.style.background_gradient(cmap="Blues", axis=None).set_caption("Routing: domain → head"))
display(pd.crosstab(rt.domain, rt.router_top).style.background_gradient(cmap="Greens", axis=None)
        .set_caption("Router's top fine-grained 'kind of text'"))

In [ ]:
misrouted = rt[rt.routed_to != expected_route][["domain", "id", "routed_to", "router_top", "router_p"]]
print("misrouted cases:"); display(misrouted.reset_index(drop=True))
cov = df[df.covered]
print(f"covered domains: original {cov.original_ok.mean():.1%} → MoE {cov.moe_ok.mean():.1%} → oracle {cov.oracle_ok.mean():.1%}"
      "  (MoE-to-oracle gap = cost of routing mistakes)")

## 8. Calibration and confidence gating
Laya is meant to be a *System 1* gate: act automatically when confident, hand the rest to a human (or a bigger model).
So beyond accuracy, we check whether each model's confidence can be trusted.

In [ ]:
def ece(conf, ok, bins=10):
    conf, ok = np.asarray(conf, float), np.asarray(ok, float)
    edges = np.linspace(0, 1, bins + 1); e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs(ok[m].mean() - conf[m].mean())
    return e

cal = pd.DataFrame({m: {"mean confidence": df[f"{m}_conf"].mean(), "accuracy": df[f"{m}_ok"].mean(),
                        "ECE (lower is better)": ece(df[f"{m}_conf"], df[f"{m}_ok"]),
                        "Brier on top answer (lower is better)": np.mean((df[f"{m}_conf"] - df[f"{m}_ok"]) ** 2)}
                    for m in ("original", "moe", "oracle")}).T
display(cal.style.format("{:.3f}"))

fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
for m, c, lab in [("original", "#9aa5b1", "Original Laya"), ("moe", "#2f6fdf", "Laya MoE")]:
    conf, ok = df[f"{m}_conf"].values.astype(float), df[f"{m}_ok"].values.astype(float)
    edges = np.linspace(0, 1, 11); xs, ys = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        sel = (conf > lo) & (conf <= hi)
        if sel.sum() >= 3: xs.append(conf[sel].mean()); ys.append(ok[sel].mean())
    axs[0].plot(xs, ys, "o-", color=c, label=lab)
    order = np.argsort(-conf); cum = np.cumsum(ok[order]) / np.arange(1, len(ok) + 1)
    axs[1].plot(np.arange(1, len(ok) + 1) / len(ok), cum, color=c, label=lab)
axs[0].plot([0, 1], [0, 1], "k:", lw=1)
axs[0].set(xlabel="confidence", ylabel="observed accuracy", title="Reliability diagram")
axs[1].set(xlabel="coverage (fraction auto-handled, most confident first)", ylabel="accuracy of auto-handled answers",
           title="Selective accuracy", ylim=(0.4, 1.02))
for a in axs: a.legend(frameon=False); a.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

for thr in (0.6, 0.7, 0.8, 0.9):
    line = f"threshold {thr:.1f}: "
    for m in ("original", "moe"):
        hi = df[df[f"{m}_conf"] >= thr]
        acc = hi[f"{m}_ok"].mean() if len(hi) else float("nan")
        line += f"   {m:8} auto-acts on {len(hi):3d}/{len(df)} ({acc:.1%} correct)"
    print(line)

## 9. Where did the MoE win and lose?
Every answer whose correctness changed between the original Laya and the MoE.

In [ ]:
flips = df[df.original_ok != df.moe_ok].copy()
flips["change"] = np.where(flips.moe_ok, "fixed by MoE", "broken by MoE")
states = {c["id"]: c["state"] for d in docs for c in d["cases"]}
flips["text"] = flips.id.map(lambda i: (states[i] if isinstance(states[i], str) else json.dumps(states[i]))[:110])
cols = ["change", "domain", "id", "question", "expected", "original_pred", "moe_pred", "routed_to", "text"]
print(flips.change.value_counts().to_string(), "\n")
display(flips.sort_values(["change", "domain"])[cols].reset_index(drop=True))

net = df.assign(net=df.moe_ok.astype(int) - df.original_ok.astype(int)).groupby(["domain", "question"]).net.sum()
print("\nnet answers gained per domain / question:")
display(net[net != 0].sort_values().to_frame("net answers gained"))

## 10. Try your own text
Edit `state` and `questions` below. The cell shows the router's decision and both models' answers side by side.

In [ ]:
state = "Agent plan: rotate the production TLS certificate and restart all web servers during business hours."
questions = {
    "needs_human": {"type": "noul", "instructions": "A human should approve this action before it runs"},
    "risk": {"type": "score", "instructions": "How risky is this action?", "criteria": ["Low", "Medium", "High", "Critical"]},
}

def show(a):
    if a["type"] == "noul":
        return f"P(yes) = {a['noul']:.3f}"
    return f"{a.get('choice', a.get('score'))}   {a['probabilities']}"

o = original.system_one(state, questions)["answers"]
m = moe.system_one(state, questions)
r = m["routing"]
print(f"router → {m['expert']}   (top kind: {r['top']}, p = {r['probabilities'][r['top']]:.2f})\n")
for q in questions:
    print(f"{q}\n  original : {show(o[q])}\n  MoE      : {show(m['answers'][q])}")

## 11. Save the results

In [ ]:
os.makedirs("results", exist_ok=True)
df.to_csv("results/laya_vs_moe_answers.csv", index=False)
rt.to_csv("results/laya_vs_moe_routing.csv", index=False)
summary.to_csv("results/laya_vs_moe_summary.csv")
print("wrote results/laya_vs_moe_answers.csv, _routing.csv, _summary.csv")
try:
    from google.colab import files  # only available in Colab
    files.download("results/laya_vs_moe_answers.csv")
except ImportError:
    pass

## Notes and caveats
* **Small eval set.** 108 cases / 312 answers; the bootstrap CI in §6 shows how much of the gain survives that. Per-domain numbers (24–48 answers each) are indicative only.
* **Expert training data is synthetic** and rule-labeled; the eval cases are hand-written and were not used for training.
* **Uncovered domains** only change when the router wrongly sends a case to an expert — that is the price of routing, visible in §7.
* The MoE runs one extra encoder pass for routing, so it costs roughly 2× the original's latency (see §4).
* Modified derivative of `laya-typed-decisions` (Apache-2.0, © ConvAI Innovations); unofficial, not affiliated.